## M0: CNN + Vanilla RNN (True Baseline)

The simplest possible CNN+RNN architecture. Uses a vanilla RNN (SimpleRNN) with no gating mechanisms as the temporal modeling component. Serves as the true baseline against which M1 (GRU), M2 (LSTM), and the transfer learning models are compared.

In [1]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1024**3:.2f} GiB")

GPU free: 10.90 GiB


In [2]:
import os, gc, time, csv, cv2
os.environ["KERAS_BACKEND"]               = "torch"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]     = "expandable_segments:True"

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import keras
from keras import layers, optimizers

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = Path("/home2/msds2026/fcapati/ml3_2026/Projects/Final_Project/data")
FRAMES_DIR  = DATA_DIR / "UCF50_frames"
DATASET_DIR = DATA_DIR / "UCF50"

# ── Config ────────────────────────────────────────────────────────────────────
IMG_HEIGHT      = 64
IMG_WIDTH       = 64
SEQUENCE_LENGTH = 16
BATCH_SIZE      = 4
EPOCHS          = 20
LEARNING_RATE   = 1e-4
NUM_CLASSES     = 50

# ── Class mapping ─────────────────────────────────────────────────────────────
CLASS_NAMES_SORTED = sorted([d.name for d in DATASET_DIR.iterdir() if d.is_dir()])
class_to_idx       = {cls: idx for idx, cls in enumerate(CLASS_NAMES_SORTED)}
idx_to_class       = {idx: cls for cls, idx in class_to_idx.items()}

print(f"✅ Keras backend : {keras.backend.backend()}")
print(f"✅ Device        : {DEVICE}")
print(f"✅ Classes       : {len(class_to_idx)}")
print(f"✅ Frames dir    : {FRAMES_DIR.exists()}")

✅ Keras backend : torch
✅ Device        : cuda
✅ Classes       : 50
✅ Frames dir    : True


In [3]:
class PreextractedDataset(Dataset):
    """Loads pre-extracted .npy frame arrays — fast, no video I/O during training."""
    def __init__(self, frames_dir, split, class_to_idx):
        self.samples = []
        for class_dir in sorted((Path(frames_dir) / split).iterdir()):
            if not class_dir.is_dir():
                continue
            label = class_to_idx[class_dir.name]
            for npy_path in sorted(class_dir.glob("*.npy")):
                self.samples.append((str(npy_path), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        npy_path, label = self.samples[idx]
        frames = np.load(npy_path)  # (16, 112, 112, 3)
        
        # Resize to target resolution if needed
        if frames.shape[1] != IMG_HEIGHT:
            frames = np.stack([
                cv2.resize(f, (IMG_WIDTH, IMG_HEIGHT)) for f in frames
            ])
        
        frames = torch.tensor(frames).permute(3, 0, 1, 2).float()
        return frames, torch.tensor(label, dtype=torch.long)


train_dataset = PreextractedDataset(FRAMES_DIR, "train", class_to_idx)
val_dataset   = PreextractedDataset(FRAMES_DIR, "val",   class_to_idx)
test_dataset  = PreextractedDataset(FRAMES_DIR, "test",  class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)

print(f"✅ Train : {len(train_dataset):,} videos → {len(train_loader)} steps/epoch")
print(f"✅ Val   : {len(val_dataset):,} videos → {len(val_loader)} steps/epoch")
print(f"✅ Test  : {len(test_dataset):,} videos → {len(test_loader)} steps/epoch")

sample_frames, sample_labels = next(iter(train_loader))
print(f"\nBatch frames : {sample_frames.shape}  → (B, C, T, H, W)")
print(f"Batch labels : {sample_labels.shape}  dtype={sample_labels.dtype}")

✅ Train : 5,288 videos → 1322 steps/epoch
✅ Val   : 640 videos → 160 steps/epoch
✅ Test  : 698 videos → 175 steps/epoch

Batch frames : torch.Size([4, 3, 16, 64, 64])  → (B, C, T, H, W)
Batch labels : torch.Size([4])  dtype=torch.int64


In [4]:
def train_model_keras(model, model_name, train_loader, val_loader, epochs=EPOCHS):
    """
    Training loop for Keras (torch backend) with PyTorch DataLoader.
    Features: per-batch cache clearing, early stopping (patience=5),
    checkpointing, CSV history logging, flush=True for real-time output.
    """
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU memory reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GiB", flush=True)

    model.compile(
        optimizer = optimizers.Adam(learning_rate=LEARNING_RATE),
        loss      = "sparse_categorical_crossentropy",
        metrics   = ["accuracy"]
    )

    best_val_loss  = float("inf")
    patience_count = 0
    patience       = 5
    history        = []

    print(f"\n{'='*60}", flush=True)
    print(f"Training: {model_name}", flush=True)
    print(f"{'='*60}", flush=True)

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        # ── Training ──────────────────────────────────────────────────────────
        train_losses, train_accs = [], []
        for batch_idx, (frames, labels) in enumerate(train_loader):
            frames_np = frames.permute(0, 2, 3, 4, 1).numpy()
            labels_np = labels.numpy()
            metrics   = model.train_on_batch(frames_np, labels_np)
            train_losses.append(metrics[0])
            train_accs.append(metrics[1])

            if (batch_idx + 1) % 50 == 0:
                torch.cuda.empty_cache()

            if (batch_idx + 1) % 100 == 0:
                print(f"  Epoch {epoch} | Batch {batch_idx+1}/{len(train_loader)} | "
                      f"loss: {metrics[0]:.4f}  acc: {metrics[1]:.4f}", flush=True)

        # ── Validation ────────────────────────────────────────────────────────
        val_losses, val_accs = [], []
        for frames, labels in val_loader:
            frames_np = frames.permute(0, 2, 3, 4, 1).numpy()
            labels_np = labels.numpy()
            metrics   = model.test_on_batch(frames_np, labels_np)
            val_losses.append(metrics[0])
            val_accs.append(metrics[1])

        train_loss = sum(train_losses) / len(train_losses)
        train_acc  = sum(train_accs)  / len(train_accs)
        val_loss   = sum(val_losses)  / len(val_losses)
        val_acc    = sum(val_accs)    / len(val_accs)
        elapsed    = time.time() - t0

        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc
        })

        print(f"Epoch {epoch:02d}/{epochs} | "
              f"train_loss: {train_loss:.4f}  train_acc: {train_acc:.4f} | "
              f"val_loss: {val_loss:.4f}  val_acc: {val_acc:.4f} | "
              f"{elapsed:.1f}s", flush=True)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model.save(f"best_{model_name}.keras")
            print(f"  💾 Saved best model (val_loss: {best_val_loss:.4f})", flush=True)
            patience_count = 0
        else:
            patience_count += 1
            print(f"  ⏳ No improvement ({patience_count}/{patience})", flush=True)
            if patience_count >= patience:
                print(f"\nEarly stopping at epoch {epoch}.", flush=True)
                break

    with open(f"history_{model_name}.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)

    print(f"\n✅ Training complete. Best val_loss: {best_val_loss:.4f}", flush=True)
    return model, history

print("✅ Training loop ready")

✅ Training loop ready


### Model Definition

A vanilla RNN computes `h_t = tanh(W_h · h_{t-1} + W_x · x_t + b)` at each timestep. Unlike GRU and LSTM, it has no gating mechanism to control information flow, making it prone to the vanishing gradient problem over longer sequences. This architectural simplicity is precisely what makes it the appropriate baseline — any performance gain in M1 and M2 can be directly attributed to the gating mechanisms.

In [5]:
# ── M0: CNN + Vanilla RNN (True Baseline) ────────────────────────────────────
# The simplest possible recurrent architecture — no gating mechanisms.
# A standard RNN cell computes: h_t = tanh(W_h * h_{t-1} + W_x * x_t + b)
# Unlike GRU/LSTM, it has no forget gate or input gate, making it prone to
# the vanishing gradient problem over long sequences. Used here as the
# simplest possible temporal modeling baseline.

inputs = keras.Input(shape=(SEQUENCE_LENGTH, IMG_HEIGHT, IMG_WIDTH, 3))

# ── TimeDistributed CNN Block (same as M1/M2) ─────────────────────────────────
x = layers.TimeDistributed(layers.Conv2D(16, (3,3), activation="relu", padding="same"))(inputs)
x = layers.TimeDistributed(layers.MaxPooling2D((2,2)))(x)

x = layers.TimeDistributed(layers.Conv2D(32, (3,3), activation="relu", padding="same"))(x)
x = layers.TimeDistributed(layers.MaxPooling2D((2,2)))(x)

x = layers.TimeDistributed(layers.Conv2D(64, (3,3), activation="relu", padding="same"))(x)
x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)

# ── Vanilla RNN Temporal Block ────────────────────────────────────────────────
# SimpleRNN is Keras's vanilla RNN implementation.
# No gating — purely: h_t = tanh(W_h * h_{t-1} + W_x * x_t + b)
x = layers.SimpleRNN(128, return_sequences=False)(x)

# ── Classification Head ───────────────────────────────────────────────────────
x = layers.Dense(256, activation="relu")(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="CNN_VanillaRNN")

total_params = sum(int(np.prod(w.shape)) for w in model.weights)
print(f"Model     : CNN_VanillaRNN")
print(f"Device    : {DEVICE}")
print(f"Params    : {total_params:,}")
model.summary()

Model     : CNN_VanillaRNN
Device    : cuda
Params    : 94,162


Model: "CNN_VanillaRNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 16, 64, 64, 3)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 16, 64, 64, 16) │           448 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 16, 32, 32, 16) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 16, 32, 32, 32) │         4,640 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 16, 16, 16, 32) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 16, 16, 16, 64) │        18,496 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, 16, 64)         │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │        12,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 94,162 (367.82 KB)

 Trainable params: 94,162 (367.82 KB)

 Non-trainable params: 0 (0.00 B)

### Training

In [ ]:
# ── Train M0 ─────────────────────────────────────────────────────────────────
model, history = train_model_keras(model, "CNN_VanillaRNN", train_loader, val_loader)

GPU memory reserved: 0.00 GiB

Training: CNN_VanillaRNN
  Epoch 1 | Batch 100/1322 | loss: 3.9240  acc: 0.0150
  Epoch 1 | Batch 200/1322 | loss: 3.9085  acc: 0.0188
  Epoch 1 | Batch 300/1322 | loss: 3.8895  acc: 0.0250
  Epoch 1 | Batch 400/1322 | loss: 3.8748  acc: 0.0281
  Epoch 1 | Batch 500/1322 | loss: 3.8510  acc: 0.0350
  Epoch 1 | Batch 900/1322 | loss: 3.7805  acc: 0.0425
  Epoch 1 | Batch 1000/1322 | loss: 3.7666  acc: 0.0440
  Epoch 1 | Batch 1100/1322 | loss: 3.7566  acc: 0.0475
  Epoch 1 | Batch 1200/1322 | loss: 3.7369  acc: 0.0504
  Epoch 1 | Batch 1300/1322 | loss: 3.7212  acc: 0.0538
Epoch 01/20 | train_loss: 3.8253  train_acc: 0.0356 | val_loss: 3.6957  val_acc: 0.0588 | 456.7s
  💾 Saved best model (val_loss: 3.6957)
  Epoch 2 | Batch 100/1322 | loss: 3.6786  acc: 0.0607
  Epoch 2 | Batch 200/1322 | loss: 3.6673  acc: 0.0624
  Epoch 2 | Batch 300/1322 | loss: 3.6590  acc: 0.0630
  Epoch 2 | Batch 400/1322 | loss: 3.6508  acc: 0.0644
  Epoch 2 | Batch 500/1322 | loss